In [1]:
# ============================================================
# TEMPORAL FUSION TRANSFORMER (TFT)
# Deep Learning for Demand Forecasting
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Deep learning imports
import torch
import pytorch_lightning as pl
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import RMSE, MAE, SMAPE

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 6)

# Set random seeds for reproducibility
pl.seed_everything(42)

print("="*70)
print("TEMPORAL FUSION TRANSFORMER (TFT)")
print("Deep Learning Model - Portfolio Showcase")
print("="*70)
print(f"\nPyTorch version: {torch.__version__}")
print(f"PyTorch Lightning version: {pl.__version__}")
print("\n✓ Libraries loaded successfully")

Seed set to 42


TEMPORAL FUSION TRANSFORMER (TFT)
Deep Learning Model - Portfolio Showcase

PyTorch version: 2.10.0+cpu
PyTorch Lightning version: 2.6.1

✓ Libraries loaded successfully


In [2]:
# ============================================================
# LOAD AND PREPARE DATA FOR TFT
# ============================================================

# Load processed data with all features
data_path = Path("../../data/processed/sample_with_features.parquet")
df = pd.read_parquet(data_path)

print(f"Dataset loaded: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

# TFT requires specific data preparation
# Create time index (continuous integer)
df = df.sort_values(['id', 'date']).reset_index(drop=True)
df['time_idx'] = df.groupby('id').cumcount()

# Create target variable
df['target'] = df['sales']

# Select features for TFT
# We'll use a subset of our engineered features
static_categoricals = ['id', 'item_id', 'store_id', 'state_id', 'cat_id', 'dept_id']
time_varying_known_categoricals = ['day_of_week', 'month', 'is_weekend']
time_varying_known_reals = ['day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos']
time_varying_unknown_reals = [
    'target',  # This is what we're predicting
    'sell_price',
    'sales_lag_7',
    'sales_lag_14',
    'sales_rolling_mean_7',
    'sales_rolling_mean_28',
    'sales_rolling_std_7',
    'sales_rolling_std_28'
]

# Remove NaN values (from lag features)
df_clean = df.dropna(subset=time_varying_unknown_reals).copy()

print(f"\nData after removing NaN: {df_clean.shape}")
print(f"Products: {df_clean['id'].nunique()}")
print(f"Time steps per product: {df_clean.groupby('id')['time_idx'].count().mean():.0f}")

# Split train/validation
max_prediction_length = 28  # Forecast 28 days ahead
max_encoder_length = 60     # Use 60 days of history

training_cutoff = df_clean['time_idx'].max() - max_prediction_length

train = df_clean[df_clean['time_idx'] <= training_cutoff].copy()
val = df_clean[df_clean['time_idx'] > training_cutoff].copy()

print(f"\nTrain set: {len(train)} rows")
print(f"Validation set: {len(val)} rows")
print("\n✓ Data prepared for TFT")

Dataset loaded: (18850, 61)
Date range: 2011-01-29 00:00:00 to 2016-03-27 00:00:00

Data after removing NaN: (18710, 63)
Products: 10
Time steps per product: 1871

Train set: 18430 rows
Validation set: 280 rows

✓ Data prepared for TFT


In [12]:
# ============================================================
# TFT IMPLEMENTATION NOTES
# ============================================================

print("""
TFT Implementation - Learning Notes
====================================

CHALLENGES ENCOUNTERED:
- Intermittent demand (68% zeros) difficult for deep learning
- Small sample size (10 products) insufficient for TFT
- Complex data requirements (encoder/decoder lengths)

LESSONS LEARNED:
- TFT works best with:
  * Hundreds/thousands of time series
  * Dense (non-zero) data
  * Long history (years of data)
  
- For this use case, simpler models (ARIMA) perform better

ALTERNATIVE APPROACHES USED:
- ARIMA: 0.7745 RMSE ✓
- Prophet: 0.7816 RMSE ✓
- Multi-product batch forecasting ✓

CONCLUSION:
TFT implementation postponed. Using proven statistical methods
for this intermittent demand scenario. TFT could be revisited
with larger, denser datasets in future iterations.
""")


TFT Implementation - Learning Notes

CHALLENGES ENCOUNTERED:
- Intermittent demand (68% zeros) difficult for deep learning
- Small sample size (10 products) insufficient for TFT
- Complex data requirements (encoder/decoder lengths)

LESSONS LEARNED:
- TFT works best with:
  * Hundreds/thousands of time series
  * Dense (non-zero) data
  * Long history (years of data)

- For this use case, simpler models (ARIMA) perform better

ALTERNATIVE APPROACHES USED:
- ARIMA: 0.7745 RMSE ✓
- Prophet: 0.7816 RMSE ✓
- Multi-product batch forecasting ✓

CONCLUSION:
TFT implementation postponed. Using proven statistical methods
for this intermittent demand scenario. TFT could be revisited
with larger, denser datasets in future iterations.

